# タンパク質相互作用解析パイプライン

BSA計算・BLAST/MSA・保存度スコアリングによる界面残基解析ツール。

2種類の解析に対応:
- **タンパク質-タンパク質/DNA界面解析**（`analyze_protein_interface`）
- **タンパク質-リガンド界面解析**（`analyze_ligand_interface`）

## 1. インポート

In [27]:
#標準ライブラリ
import os
import subprocess
import shutil
import copy

#データ処理
import pandas as pd  #データ処理、CSV出力用
import numpy as np
import math

#Biopython関連
from Bio.PDB import PDBList, PDBParser  #PDBファイルのダウンロードと構造解析用
from Bio.PDB.SASA import ShrakeRupley  #溶媒接近可能表面積(SASA)の計算用
from Bio.PDB.Polypeptide import PPBuilder #FASTAファイル作成用
from Bio.Blast import NCBIWWW #BLAST検索
from Bio.Blast import NCBIXML
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
from Bio import AlignIO
from Bio.PDB import PDBIO #pymol用

## 2. 共通ヘルパー関数

SASA計算・結合種別判定。両方の解析モードから共通で使われる。

In [28]:
#SASA計算関数の定義
def get_residue_sasa(structure):
    """
    構造内の全残基のSASAを計算し、辞書形式で返します。
    Args:
        structure (Bio.PDB.Structure): 解析対象のPDB構造オブジェクト  
    Returns:
        dict: {(chain_id, res_no): sasa_value} の形式の辞書
    """
    #1.SASA計算用の計算機(ShrakeRupley)を用意
    sr = ShrakeRupley()

    # 2. 計算を実行。level="R" とすることで、原子ごとではなく、
    # 各「残基(Residue)」が持つ全原子の面積を合計した値を出す。
    sr.compute(structure, level="R") 
    
    # 3. 結果を保存するための空の辞書を用意。
    # あとで「特定の残基の面積」をIDで取り出すため。
    sasa_dict = {}

    # 4. 構造から残基を一つずつ取り出してループ
    for res in structure.get_residues():
        # 残基のフルIDを取得。
        # full_id の中身は (PDB名, モデル番号, チェーン名, (ヘテロ, 残基番号, 挿入符号))
        full_id = res.get_full_id() # (pdb_id, model, chain, res_no)
        
        # (チェーン名, 残基番号) という「タプル」を辞書の「キー」にする。
        # チェーン名：full_id[2]  （例: 'M'）
        # 残基番号：full_id[3][1] （例: 506）
        res_key = (full_id[2], full_id[3][1])

        # 計算機によって各残基オブジェクト（res）に書き込まれた .sasa の値を、
        # さっき作った ID（res_key）に対応させて辞書(sasa_dict)にメモする。
        sasa_dict[res_key] = res.sasa
    return sasa_dict


# 相互作用の種類の判定関数
def classify_interaction(row):
    """
    原子間の距離と種類に基づき、結合のタイプを分類します。

    Args:
        row (dict): 1対の原子接触データが入った「辞書」。
                    キーとして 'Min_Atom_Dist', 'Atom1', 'Atom2' を持つ。
    Returns:
        str: 判定された結合の名前（文字列）
    """
    #辞書(row)のキーに指定して、判定に必要な値を抽出
    dist = row['Min_Atom_Dist']
    a1 = row['Atom1']
    a2 = row['Atom2']
    
    # (1)塩橋、水素結合  
    # 3.5Å以下かつ、DNAのリン酸基(OP1/OP2)と、タンパク質の親水性原子(O,N)の近接
    if dist < 3.5:
        if ('O' in a1 or 'N' in a1) and ('OP' in a2 or 'O' in a2):
            return "Hydrogen Bond / Salt Bridge"
        
    # (2)疎水性相互作用
    # 4.5Å以下かつ、炭素同士の近接
    if dist < 4.5:
        if 'C' in a1[0] and 'C' in a2[0]:
            return "Hydrophobic Interaction"

    #上記に該当しない近接は、ファンデルワールス力とする。       
    return "Van der Waals / Close Contact"

## 3. 配列抽出・BLAST検索・保存度スコアリング（共通ユーティリティ）

こちらも両方の解析モードから共通で使われる。

In [29]:
#指定したチェーン配列を抽出し、FASTA形式でファイルに保存する。
def save_fasta_from_structure(pdb_id, structure, chain_ids, output_file):
    
    ppb = PPBuilder()
    fasta_records = []

    print(f"---{pdb_id}から配列を抽出中---")

    #output_fileを書き込みモード(w)で開く
    with open(output_file, "w") as f:
        #引数で受け取ったchain_idsから、要素を一つずつ取り出してcidに代入し、ループ処理
        for cid in chain_ids:
            #指定したチェーンを取得
            chain = structure[0][cid]

            #PPBuildeを使って、構造からポリペプチドを構築
            #途中で欠損がある場合は、複数の断片としてリストに返ってくる
            peptides = ppb.build_peptides(chain)

            #全断片を結合して一つの配列にする
            #断片化されてるかもしれない配列をつなげるための、空の文字列を用意
            full_seq = ""
            #見つかった断片(pp)を一つずつループし、get_sequence() でアミノ酸配列（Seqオブジェクト）に変換。
            #さらに str() で普通の文字列にして、full_seq に後ろから連結。
            for pp in peptides:
                full_seq += str(pp.get_sequence())
            
            #配列が空ではない場合(正しく抽出できた)に、以下の書き込み処理を行う。
            if full_seq:
                print(f"Chain {cid}:{len(full_seq)}amino acids")
                #FASTA形式でファイルに書き込み。
                f.write(f">{pdb_id}_{cid}\n{full_seq}\n")
                #リストにも配列を追加。関数の戻り値用。
                fasta_records.append(full_seq)
            else:
                print(f"  警告: Chain {cid} から配列を抽出できませんでした。")
        
        print(f"★FASTAファイルを保存しました: {output_file}")
        return fasta_records, structure

#FASTAからBLAST検索をしてXMLで保存
def run_blast_and_save_xml(fasta_file, output_xml):
    """
    FASTAファイルを読み込み、BLAST検索を実行して結果をXMLで保存する。
    すでにXMLが存在する場合は検索をスキップする（キャッシュ機能）。
    """
    #すでにxmlの結果ファイルがあるか確認。(サーバー接続に時間がかかるため二度手間防止)
    if os.path.exists(output_xml):
        print(f"すでに{output_xml}が存在するため、BLAST検索をスキップします。") 
        return
    
    #FASTAファイルから配列を読み込む
    with open (fasta_file, "r") as f:
        fasta_string = f.read()
    
    print(f"--- BLAST検索を開始します（NCBIサーバーへ接続中...） ---")

    try:
        #qblast関数の実行
        # program="blastp": タンパク質配列の検索
        # database="nr": 世界最大の非冗長タンパク質データベース
        # expect=0.001: 期待値（E-value）のしきい値。低いほど厳選される。
        result_handle = NCBIWWW.qblast(
            program="blastp", 
            database="nr", 
            sequence=fasta_string,
            expect=0.001,
            hitlist_size=200  # 似た配列を最大200件取得
        )

        # 結果（XML形式）をファイルに保存
        with open(output_xml, "w") as out_f:
            out_f.write(result_handle.read())
        
        result_handle.close()
        print(f"★ BLAST検索が完了し、結果を保存しました: {output_xml}")

    except Exception as e:
         print(f"エラーが発生しました: {e}")

#マルチプルアライメント(MSA)実施前に、BLSTA検索したXMLファイルをFASTAに変換。
def extract_hits_to_fasta(xml_file, output_fasta, query_fasta):
    """
    BLASTのXML結果からHitした配列を抜き出し、
    クエリ配列と合わせて1つのFASTAファイルにする。
    """
    try:
        # クエリ配列を読み込み、リスト(records)にする。
        # SeqIO.parsehaは配列読み込みのスキャナーの役割。
        records = list(SeqIO.parse(query_fasta, "fasta"))
        query_id = records[0].id

        #XMLを開いてHit配列を抜き出す
        with open(xml_file) as result_handle:
            blast_records = NCBIXML.parse(result_handle)

            #取得したXMLをループ処理。1つ目はクエリ。2つ目はヒットした相手。
            for blast_record in blast_records:
                for alignments in blast_record.alignments:
                    #最もよく似ているメインの断片を取得。
                    hsp = alignments.hsps[0]

                    #Hit配列をSeqRecordオブジェクトに変換。
                    #文字列であるhsp.hseqを、Seq()に入れて、biopythonにさせる。
                    #ヒット配列のアクセッション番号をIDに設定。配列についての説明文を設定。
                    rec = SeqRecord(
                    Seq(hsp.sbjct),
                    id=alignments.accession,
                    description=alignments.hit_def
                    )
                    #今作成した箱(rec)を、リスト(records)の末尾に追加。
                    records.append(rec)

        SeqIO.write(records, output_fasta, "fasta")
        print(f"★ MSA用の入力ファイルを作成しました: {output_fasta} ({len(records)}件)")     
    
    except Exception as e:
        print(f"【extract_hits_to_fasta内でエラーが発生】: {e}")
        raise e

def calculate_shannon_entropy(column_seq):
    """
    1行分のアミノ酸配列を受け取り、保存度スコア(0-1)を返す
    """
    #アライメントの縦列からギャップ(-)を除いて、アミノ酸リストを作成する。(リスト内包表記)
    observations = [res for res in column_seq if res != "-"]
    #アミノ酸数をカウント。
    n_obs = len(observations)

    #アライメントが全部ギャップなら、0を返す。
    if n_obs == 0:
        return 0.0
    
    #出現したアミノ酸種類別に、個数をカウントする
    unique_elemnts, counts = np.unique(observations, return_counts=True)
    #トータルアミノ酸数におけるアミノ酸種類ごとの割合を示す。
    probs = counts / n_obs
    
    # シャノン・エントロピーの計算
    entropy = -sum([p*math.log(p, 2) for p in probs])

    # 最大エントロピー(log2(20) ≒ 4.32)で正規化し、保存度スコアに反転
    # 1.0 = 完全に保存、0.0 = バラバラ
    max_entropy = math.log(20, 2)
    conservation = 1.0 - (entropy / max_entropy)

    # 計算結果のマイナス値を0に補正し、小数点第4位で整えて返す
    max_conservation = max(0, conservation) # マイナスを0に直す
    if max_conservation != 0:
        return round(max_conservation, 4)   # 四捨五入して返す
    else:
        return 0.0                          # 0なら0.0を返す

def integrate_conservation_to_csv(msa_output_fasta, csv_name, output_csv, query_id):
    """
    MSAの結果から保存度を算出し、既存のCSVに統合する。
    """
    # MSAファイルを読み込む
    try:
        alignment = AlignIO.read(msa_output_fasta, "fasta")
    except Exception as e:
        print(f"MSAファイル読み込みエラー: {e}")
        return

    # クエリ配列の特定
    query_index = -1
    for i, record in enumerate(alignment):
        if query_id in record.id:
            query_index = i
            break

    if query_index == -1:
        print(f"エラー: MSA内にクエリID {query_id} が見つかりません。")
        return

    # 列ごとに保存度を計算
    conservation_data = []
    interface_df = pd.read_csv(csv_name)
    # 配列の「何番目の文字か」をカウントする変数
    # PPBuilderで抽出した配列は、構造上のアミノ酸の並びと一致させる
    res_index_in_seq = 0 

    # PDBの開始残基番号を取得
    start_res_no = interface_df['Res1_No'].min()

    for col in range(alignment.get_alignment_length()):
        column_seq = alignment[:, col]
        query_res = column_seq[query_index]

        # 自分がギャップでない ＝ その場所はPDB構造（クエリ）に存在する残基
        if query_res != "-":
            # 1. 保存度を計算
            score = calculate_shannon_entropy(column_seq)
            
            # 2. PDB上の正しい残基番号を計算
            # 1文字目なら start_res_no + 0、2文字目なら + 1... と計算
            actual_res_no = start_res_no + res_index_in_seq
            
            conservation_data.append({
                'Res1_No': actual_res_no,
                'Conservation_Score': score
            })
            
            # クエリの文字が見つかったので、インデックスを1進める
            res_index_in_seq += 1

    # 計算結果をDataFrameに変換
    conservation_df = pd.DataFrame(conservation_data)

    # 既存のCSVとマージ
    final_df = pd.merge(interface_df, conservation_df, on='Res1_No', how='left')
    
    final_df.to_csv(output_csv, index=False)
    print(f"★ 統合完了 保存先: {output_csv}")

## 4. タンパク質-タンパク質/DNA界面解析

2本のポリマー鎖（タンパク質同士、またはタンパク質とDNA）の界面を解析する。

In [30]:
def analyze_protein_interface(pdb_id, chain_id1, chain_id2, dna_list, threshold=15.0):
    print(f"--- {pdb_id} の解析を開始 ---")

    # PDBファイルの取得
    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir='.', file_format='pdb')

    # 構造読み込み
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_file)

    # チェーンの取得
    model = structure[0]





#SASAの算出
    # 1. 複合体(Protein1 + (Protein2 or DNA))の状態でのSASA計算
    #Protein2 or DNAがいる状態で、どれくらい表面が露出しているか辞書(sasa_complex)にメモする。
    sasa_complex = get_residue_sasa(structure)

    # 2. Protein1単体の状態でのSASA計算する。
    # Protein2 or DNAを取り除いたモデルを一時的に作成するために、構造をコピー(deepcopy)する
    import copy
    protein_only = copy.deepcopy(structure)

    # コピーした構造(protein_only)の中身を、一つずつ確認する
    for model_copy in protein_only:

        #Protein2 or DNAが複数ある場合を想定し、ループ処理。
        #dna_listは、解析時に指定した['I', 'J']のようなリスト
        for dna_id in dna_list:
            
            #もしモデル内にDNAがあったら、DNAを削除
            if dna_id in model_copy:
                model_copy.detach_child(dna_id)
    
    #Protein2 or DNAを削除した構造(protein_only)を、冒頭作成した関数を使って、
    #別の辞書(sasa_isolated)にメモする。
    sasa_isolated = get_residue_sasa(protein_only)


    #接触を確認したいチェーン2つをc1,c2とする。
    c1 = model[chain_id1]
    c2 = model[chain_id2]

    #解析結果をためる空のリストを用意
    results = []

    #チェーン1の残基を一つずつ取り出してループ
    for res1 in c1.get_residues():
        #CA原子がない残基は飛ばす。
        if 'CA' not in res1: continue

        #---この残基の埋没表面積(BSA)計算---
        # BSA = (チェーン1が埋没してないときの面積) - (チェーン2で隠れたチェーン1の面積)
        # 辞書(sasa_isolated, sasa_complex)から
        # (チェーン名、残基番号)をキーにして面積を取得する。
        # データがなくてもエラーにせず、「面積0」として進める(get.(,0)の機能)。
        s_iso = sasa_isolated.get((chain_id1, res1.id[1]), 0) #Protein1単体時の表面積
        s_com = sasa_complex.get((chain_id1, res1.id[1]), 0) #Protein2で隠されたときのProtein1の表面積
        # 誤差等でマイナスにならないよう、max(0,)。少数第二位で四捨五入。
        res_bsa = round(max(0, s_iso - s_com), 2) 


        #原子間距離計算
        #チェーン2を一つずつ取り出してループ
        for res2 in c2.get_residues():
            # DNAなら 'P'、タンパク質なら 'CA' を基準にする
            # どちらもある場合は、タンパク質（CA）を優先
            if 'CA' in res2:
                target_atom = 'CA'
            elif 'P' in res2:
                target_atom = 'P'
            else:
                continue # どちらもなければスキップ
            
            #chain1と、chain2の残基を引き算して、中心同士の距離を求める。
            dist = res1['CA'] - res2[target_atom]

            #1.ざっくり判定
            #中心同士の残基距離が、15Å以下のペアを抽出。
            #ある程度絞ることで計算時間を短縮する。
            if dist < threshold:
                
                #2.詳細判定
                #最小残基距離を記録するために、まずはありえない数値999Åを初期値として設定する。
                min_atom_dist = 999
                best_pair = ("","")

                #残基1から全原子を一つずつ取り出してループ。
                for atom1 in res1.get_atoms():
                    #残基2から全原子を一つずつ取り出してループ。
                    for atom2 in res2.get_atoms():

                        #原子1と原子2の中心同士の距離をa_distとして求める。
                        a_dist = atom1 - atom2

                        # 今までの最短距離(min_atom_dist)よりも近いペアが見つかったら、データを更新
                        if a_dist < min_atom_dist:
                            # この時点での最短距離を上書き保存
                            min_atom_dist = a_dist  
                            # その時の原子名の組み合わせ（例：'N' と 'OP1'）を記録
                            best_pair = (atom1.get_name(), atom2.get_name())
            

            #【今回変更】ついでに、結合種類をappendに組み込んだ。
            #df作成前のため、手元に生の変数しかない。どの変数が、Atom1かを指定する必要あり。
                
                #結合の判定をする
                #最短距離、原子1、原子2を辞書(冒頭で定義したrowに該当)にメモする。
                # 冒頭で作った判定関数(classify_interaction)に渡し、結合名を受け取る。
                current_interaction = classify_interaction({
                    'Min_Atom_Dist': min_atom_dist,
                    'Atom1': best_pair[0],
                    'Atom2': best_pair[1]
                    })
                
                #空のリスト(results)に情報を保存
                #情報をひとまとめに、一行データとして放り込む(append)
                results.append({
                    'PDB_ID': pdb_id,
                    'Chain1': chain_id1,
                    'Res1_No': res1.id[1],
                    'Res1_Name': res1.get_resname(),
                    'Chain2': chain_id2,
                    'Res2_No': res2.id[1],
                    'Res2_Name': res2.get_resname(),
                    'Min_Atom_Dist': round(float(min_atom_dist), 2), #小数点第二位で四捨五入
                    'Atom1':best_pair[0],
                    'Atom2':best_pair[1],
                    'BSA':res_bsa,
                    'Interaction_Type':current_interaction
                })

    
    if results:
        #リスト(results)をdataframeに整える
        df = pd.DataFrame(results)

        # 合計BSAの算出（重複なしで計算）
        # chain1の1つの残基が、chain2の複数残基と接触してる場合、 BSAが重複してカウントされるので、
        # 同じ残基(Chain1, Res1_No)の重複(drop_duplicates)を除いた上で、合計を出す。
        unique_res_df = df.drop_duplicates(subset=['Chain1', 'Res1_No'])
        #重複を消したリストから、BSAの列だけを合計する。
        total_bsa = unique_res_df['BSA'].sum()

        # 特定の残基のみ(chain1のArg)をカウント
        #dataframeからes1_Nameが、ARGのみを絞り込む
        #(メモ)1つ目のunique_res_dfは、「抜き出し元の表」を指定。2つは、「判定に使う列」を指定。
        arg_residues = unique_res_df[unique_res_df['Res1_Name']=='ARG']
        #絞り込んだ結果の「行の数」を数える(len)
        num_arg = len(arg_residues)

        # 「界面残基」の定義（BSAが1.0以上のものをカウント）
        interface_residues = unique_res_df[unique_res_df['BSA'] > 1.0]
        num_interface_res = len(interface_residues)
        
        #画面上への出力
        # f"{変数:.2f}" は、浮動小数点数を「小数点第2位まで」表示する指定
        print(f"  > 合計BSA: {total_bsa:.2f} Å²")
        print(f"  > 界面残基数 (BSA > 1.0): {num_interface_res} 残基")
        print(f"アルギニンの数:{num_arg}")
        
        #CSVファイル名の指定
        filename = f"interface_{pdb_id}_{chain_id1}_{chain_id2}.csv"
        #CSVファイル出力。index=False は、左端の行番号を保存しない設定。
        df.to_csv(filename, index=False)
        #ここまでの解析がワークしたら、「成功」を表示し、ファイル名を返す。
        print(f"成功: {len(df)}件の接点を {filename} に保存しました。")
        return structure, filename, pdb_file
    else:
        #解析が失敗したら、「警告」を表示
        print(f"警告: {pdb_id} で接点は見つかりませんでした。")
        #「何もなし」という印に、Noneを返す
        return None, None, None

## 5. タンパク質-リガンド界面解析

タンパク質チェーンと、その中の低分子リガンド(HETATM)との接触残基を解析する。
`get_residue_sasa`・`classify_interaction`（セル2）を再利用し、
`integrate_conservation_to_csv`（セル3）などの下流関数もそのまま使える。

In [31]:
def find_ligand_residues(structure, chain_id, exclude_water=True):
    """指定チェーン内のヘテロ残基（水を除く）を一覧表示するヘルパー。
    リガンドのresname・残基番号を確認するために先に実行することを推奨。
    """
    model = structure[0]
    chain = model[chain_id]
    hetero_residues = []
    for res in chain.get_residues():
        het_flag = res.id[0]
        if het_flag == ' ':
            continue  # 通常のアミノ酸残基はスキップ
        if exclude_water and het_flag == 'W':
            continue  # 水分子はスキップ
        hetero_residues.append(res)
        print(f"  検出: resname={res.get_resname()}, "
              f"res_no={res.id[1]}, 原子数={len(list(res.get_atoms()))}")
    return hetero_residues

def analyze_ligand_interface(pdb_id, chain_id, ligand_resname=None, threshold=15.0):
    """タンパク質チェーンと、その中の低分子リガンド(HETATM)との接触残基を解析する。
    analyze_protein_interface() と同じ出力形式(CSV)を返すため、
    integrate_conservation_to_csv() など下流の関数はそのまま再利用できる。

    Args:
        pdb_id (str): PDB ID (例: '3MXF')
        chain_id (str): 解析対象のタンパク質チェーンID (例: 'A')
        ligand_resname (str or None): リガンドの3文字コード。
            Noneの場合、水以外のヘテロ残基のうち原子数が最も多いものを自動選択する。
        threshold (float): 粗判定用の残基間距離の閾値(Å)

    Returns:
        (structure, csv_filename, pdb_file_path) — analyze_protein_interface()と同じ形式
    """
    print(f"--- {pdb_id} のリガンド接触残基解析を開始 ---")

    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir='.', file_format='pdb')

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_file)
    model = structure[0]
    chain = model[chain_id]

    # ===== リガンド残基の特定 =====
    hetero_residues = [res for res in chain.get_residues() if res.id[0] not in (' ', 'W')]
    if not hetero_residues:
        print(f"警告: チェーン{chain_id}にヘテロ残基(リガンド候補)が見つかりません。")
        return None, None, None

    if ligand_resname is not None:
        candidates = [r for r in hetero_residues if r.get_resname() == ligand_resname]
        if not candidates:
            print(f"警告: resname={ligand_resname} のリガンドが見つかりません。")
            return None, None, None
        ligand_res = candidates[0]
    else:
        # 原子数が最も多いヘテロ残基をリガンドとみなす（補酵素の小さいイオン等を除外する狙い）
        ligand_res = max(hetero_residues, key=lambda r: len(list(r.get_atoms())))

    print(f"  リガンドとして選択: {ligand_res.get_resname()} "
          f"(res_no={ligand_res.id[1]}, 原子数={len(list(ligand_res.get_atoms()))})")

    # ===== SASA計算（複合体 vs リガンド除去後） =====
    sasa_complex = get_residue_sasa(structure)

    protein_only = copy.deepcopy(structure)
    for model_copy in protein_only:
        target_chain = model_copy[chain_id]
        # チェーンごとではなく、リガンド残基だけを残基単位で除去
        target_chain.detach_child(ligand_res.id)
    sasa_isolated = get_residue_sasa(protein_only)

    # ===== 残基ごとのBSA・接触判定 =====
    results = []
    for res1 in chain.get_residues():
        if res1.id[0] != ' ':
            continue  # ヘテロ残基（リガンド自身や水）は対象外、タンパク質残基のみ
        if 'CA' not in res1:
            continue

        s_iso = sasa_isolated.get((chain_id, res1.id[1]), 0)
        s_com = sasa_complex.get((chain_id, res1.id[1]), 0)
        res_bsa = round(max(0, s_iso - s_com), 2)

        # 中心間距離でざっくり絞り込み（リガンドの重心的な代表原子として最初の原子を使用）
        rep_atom = next(iter(ligand_res.get_atoms()))
        dist = res1['CA'] - rep_atom
        if dist >= threshold:
            continue

        # 原子間の最短距離を精密探索
        min_atom_dist = 999
        best_pair = ("", "")
        for atom1 in res1.get_atoms():
            for atom2 in ligand_res.get_atoms():
                a_dist = atom1 - atom2
                if a_dist < min_atom_dist:
                    min_atom_dist = a_dist
                    best_pair = (atom1.get_name(), atom2.get_name())

        current_interaction = classify_interaction({
            'Min_Atom_Dist': min_atom_dist,
            'Atom1': best_pair[0],
            'Atom2': best_pair[1],
        })

        results.append({
            'PDB_ID': pdb_id,
            'Chain1': chain_id,
            'Res1_No': res1.id[1],
            'Res1_Name': res1.get_resname(),
            'Chain2': f"LIG_{ligand_res.get_resname()}",
            'Res2_No': ligand_res.id[1],
            'Res2_Name': ligand_res.get_resname(),
            'Min_Atom_Dist': round(float(min_atom_dist), 2),
            'Atom1': best_pair[0],
            'Atom2': best_pair[1],
            'BSA': res_bsa,
            'Interaction_Type': current_interaction,
        })

    if not results:
        print(f"警告: {pdb_id} でリガンドとの接点は見つかりませんでした。")
        return None, None, None

    df = pd.DataFrame(results)
    total_bsa = df['BSA'].sum()
    num_interface_res = len(df[df['BSA'] > 1.0])

    print(f"  > 合計BSA: {total_bsa:.2f} Å²")
    print(f"  > リガンド接触残基数 (BSA > 1.0): {num_interface_res} 残基")

    filename = f"ligand_interface_{pdb_id}_{chain_id}_{ligand_res.get_resname()}.csv"
    df.to_csv(filename, index=False)
    print(f"成功: {len(df)}件の接点を {filename} に保存しました。")

    return structure, filename, pdb_file

## 6. PyMOL可視化用PDB出力

In [32]:
def save_conservation_to_pdb(input_pdb, integrated_csv, output_pdb, target_chain):
    """
    CSVから保存度スコアを読み込み、PDBのB-factorカラムに上書き保存します。
    """
    # 1. データの読み込み
    df = pd.read_csv(integrated_csv)
    # 残基番号と保存度スコアの対応辞書を作成
    # 重複を除去してマッピング
    score_map = dict(zip(df['Res1_No'], df['Conservation_Score']))

    # 2. PDB構造の解析
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein', input_pdb)

    # 3. B-factorの書き換え
    for model in structure:
        for chain in model:
            if chain.id == target_chain:
                for residue in chain:
                    res_no = residue.id[1]
                    # CSVにデータがある場合はその値を、ない場合は0をセット
                    score = score_map.get(res_no, 0.0)
                    for atom in residue:
                        atom.set_bfactor(score)

    # 4. 新しいPDBとして保存
    io = PDBIO()
    io.set_structure(structure)
    io.save(output_pdb)
    print(f"★ PyMOL可視化用PDBを作成しました: {output_pdb}")

def generate_pymol_script_ligand(pdb_filename, csv_file, output_pml, target_chain, ligand_resname='JQ1'):
    """BSA（結合の強さ）に応じて接触残基を色分けするPyMOLスクリプトを生成"""

    df = pd.read_csv(csv_file)
    # 実際に接触している残基だけに絞る（BSA>1.0）、重複除去
    contact_df = df[df['BSA'] > 1.0].drop_duplicates(subset=['Res1_No'])
    max_bsa = contact_df['BSA'].max() if not contact_df.empty else 1.0

    commands = [
        f"load {pdb_filename}, structure",
        "hide everything",
        f"show cartoon, chain {target_chain}",
        f"color gray80, chain {target_chain}",   # ベースは灰色（保存度でなくBSAで語るため）
        f"show sticks, resn {ligand_resname}",
        f"color yellow, resn {ligand_resname}",
    ]

    # BSAの大きさに応じて3段階に色分け（濃い赤 = 結合が強い）
    for _, row in contact_df.iterrows():
        resi = row['Res1_No']
        bsa = row['BSA']
        ratio = bsa / max_bsa
        if ratio >= 0.6:
            color = "red"
        elif ratio >= 0.3:
            color = "orange"
        else:
            color = "salmon"

        sel = f"chain {target_chain} and resi {resi}"
        commands.append(f"show sticks, {sel}")
        commands.append(f"color {color}, {sel}")

    commands += [
        f"zoom resn {ligand_resname}, 8",
        "bg_color white",
        "set ray_opaque_background, 0",
        f"png {pdb_filename.replace('.pdb', '.png')}, width=1200, height=900, dpi=300, ray=1",
    ]

    with open(output_pml, "w") as f:
        f.write("\n".join(commands))
    print(f"★ PyMOLスクリプトを作成しました: {output_pml}")




def generate_pymol_script_protein(pdb_filename, csv_file, output_pml, chain1, chain2, max_dist=5.0):
    """相互作用の種類ごとに色分けした距離線(dash)を引くPyMOLスクリプトを生成"""

    df = pd.read_csv(csv_file)
    if df.empty:
        print(f"警告: {csv_file} が空のため、PyMOLスクリプトは作成されません。")
        return

    commands = [
        f"load {pdb_filename}, structure",
        "hide everything",
        "show cartoon",
        f"color skyblue, chain {chain1}",
        f"color palegreen, chain {chain2}",
        "set dash_gap, 0.2",
        "set dash_width, 3",
    ]

    interaction_styles = {
        "Hydrogen Bond / Salt Bridge": "yellow",
        "Hydrophobic Interaction": "red",
        "Van der Waals / Close Contact": "gray80"
    }

    for i, row in df.iterrows():
        if row['Min_Atom_Dist'] > max_dist:
            continue

        sel1 = f"(chain {row['Chain1']} and resi {row['Res1_No']} and name {row['Atom1']})"
        sel2 = f"(chain {row['Chain2']} and resi {row['Res2_No']} and name {row['Atom2']})"
        dist_name = f"dist_{row['Res1_No']}_{row['Res2_No']}_{i}"

        commands.append(f"distance {dist_name}, {sel1}, {sel2}")
        color = interaction_styles.get(row['Interaction_Type'], "white")
        commands.append(f"color {color}, {dist_name}")
        commands.append(f"show sticks, (chain {row['Chain1']} and resi {row['Res1_No']})")
        commands.append(f"show sticks, (chain {row['Chain2']} and resi {row['Res2_No']})")

    commands += [
        "hide labels",
        "deselect",
        "bg_color white",
        "set ray_opaque_background, 0",
        f"png {pdb_filename.replace('.pdb', '.png')}, width=1200, height=900, dpi=300, ray=1",
    ]

    with open(output_pml, "w") as f:
        f.write("\n".join(commands))
    print(f"★ PyMOLスクリプトを作成しました: {output_pml}")

## 7. 実行：タンパク質-リガンド版（BD1ドメイン / PDB 3MXF）

In [ ]:
# --- 実行設定（タンパク質-リガンド版） ---
# BD1ドメイン(3MXF)のリガンド接触残基を解析する
targets = [
    {'id': '3MXF', 'chain': 'A', 'ligand_resname': 'JQ1'}
]

# ループに入る前にMAFFTのパスを1回だけ確認しておく
# ①ノートブックとbinが同じ階層（ローカル作業用） ②ノートブックがscripts/の中、binが1つ上（Git構成）の両方に対応

current_dir = os.getcwd()
candidate_paths = [
    os.path.join(current_dir, "bin", "mafft-win", "mafft.bat"),        # ① ノートブックとbinが同じ階層（ローカル作業用）
    os.path.join(current_dir, "..", "bin", "mafft-win", "mafft.bat"),  # ② ノートブックがscripts/の中、binが1つ上（Git構成）
]

mafft_path = None
for p in candidate_paths:
    abs_p = os.path.abspath(p)
    if os.path.exists(abs_p):
        mafft_path = abs_p
        break

if mafft_path is None:
    print("致命的エラー: MAFFTの実行ファイルが見つかりません。以下を確認しました:")
    for p in candidate_paths:
        print(f"  - {os.path.abspath(p)}")
else:
    print(f"MAFFT確認OK: {mafft_path}")

    print("=== リガンド接触残基の一括解析を開始します ===")

    for t in targets:
        try:
            structure, csv_name, pdb_path = analyze_ligand_interface(
                t['id'], t['chain'], ligand_resname=t['ligand_resname']
            )

            if structure is not None:
                fasta_name = f"seq_{t['id']}_{t['chain']}.fasta"
                save_fasta_from_structure(t['id'], structure, [t['chain']], fasta_name)

                blast_xml_name = f"blast_{t['id']}_{t['chain']}.xml"
                run_blast_and_save_xml(fasta_name, blast_xml_name)

                msa_input_fasta = f"msa_{t['id']}_{t['chain']}.fasta"
                extract_hits_to_fasta(blast_xml_name, msa_input_fasta, fasta_name)

                msa_output_fasta = f"aligned_{t['id']}_{t['chain']}.fasta"
                input_path = os.path.abspath(msa_input_fasta)
                print(f"--- {t['id']} の整列(MSA)を実行中... ---")
                cmd = [mafft_path, "--auto", input_path]

                with open(msa_output_fasta, "w") as out_f:
                    result = subprocess.run(
                        cmd, stdout=out_f, stderr=subprocess.PIPE, text=True, check=True
                    )
                print(f"DONE: {t['id']} の整列が完了しました！ -> {msa_output_fasta}")

                final_output_csv = f"final_interface_{t['id']}_{t['chain']}.csv"
                query_id_in_fasta = f"{t['id']}_{t['chain']}"
                integrate_conservation_to_csv(
                    msa_output_fasta=msa_output_fasta,
                    csv_name=csv_name,
                    output_csv=final_output_csv,
                    query_id=query_id_in_fasta
                )

                viz_pdb_name = f"viz_{t['id']}_{t['chain']}_cons.pdb"
                save_conservation_to_pdb(
                    input_pdb=pdb_path,
                    integrated_csv=final_output_csv,
                    output_pdb=viz_pdb_name,
                    target_chain=t['chain']
                )

                viz_pml_name = f"viz_{t['id']}_{t['chain']}_cons.pml"
                generate_pymol_script_ligand(
                    pdb_filename=viz_pdb_name,
                    csv_file=final_output_csv,
                    output_pml=viz_pml_name,
                    target_chain=t['chain'],
                    ligand_resname=t['ligand_resname']
                )

                print(f"DONE: {t['id']} の全工程が正常に完了しました！")

        except subprocess.CalledProcessError as e:
            print(f"MAFFT実行エラー ({t['id']}): {e.stderr}")
        except Exception as e:
            print(f"ERROR: {t['id']}: {e}")

    print("=== すべての工程が完了しました ===")

MAFFT確認OK: C:\0 bioinfo\data\260322_ConservedSeq\260405_MAFFT\mafft-win\mafft.bat
=== リガンド接触残基の一括解析を開始します ===
--- 3MXF のリガンド接触残基解析を開始 ---
Structure exists: '.\pdb3mxf.ent' 
  リガンドとして選択: JQ1 (res_no=1, 原子数=31)
  > 合計BSA: 220.91 Å²
  > リガンド接触残基数 (BSA > 1.0): 14 残基
成功: 66件の接点を ligand_interface_3MXF_A_JQ1.csv に保存しました。
---3MXFから配列を抽出中---
Chain A:127amino acids
★FASTAファイルを保存しました: seq_3MXF_A.fasta
すでにblast_3MXF_A.xmlが存在するため、BLAST検索をスキップします。
★ MSA用の入力ファイルを作成しました: msa_3MXF_A.fasta (201件)
--- 3MXF の整列(MSA)を実行中... ---
DONE: 3MXF の整列が完了しました！ -> aligned_3MXF_A.fasta
★ 統合完了 保存先: final_interface_3MXF_A.csv
★ PyMOL可視化用PDBを作成しました: viz_3MXF_A_cons.pdb
★ PyMOLスクリプトを作成しました: viz_3MXF_A_cons.pml
DONE: 3MXF の全工程が正常に完了しました！
=== すべての工程が完了しました ===


## 8. 実行：タンパク質-タンパク質/DNA版（今後使用、例: BAF複合体プロジェクト）

In [ ]:
# --- 実行設定（タンパク質-タンパク質/DNA版） ---
# 今後（例: BAF複合体プロジェクト）で使うときは、ここにPDB ID・チェーン・
# dna_listを指定してください。使わない間は targets = [] のままで構いません。
targets = [
    # 例: {'id': '1A1A', 'c1': 'A', 'c2': 'B', 'dna_list': []},
]

if not targets:
    print("targetsが空です。タンパク質-タンパク質/DNA解析を実行する際は、上のtargetsに追加してください。")
else:
    current_dir = os.getcwd()
    candidate_paths = [
        os.path.join(current_dir, "bin", "mafft-win", "mafft.bat"),
        os.path.join(current_dir, "..", "bin", "mafft-win", "mafft.bat"),
    ]

    mafft_path = None
    for p in candidate_paths:
        abs_p = os.path.abspath(p)
        if os.path.exists(abs_p):
            mafft_path = abs_p
            break

    if mafft_path is None:
        print("致命的エラー: MAFFTがPATHに見つかりません。処理を中断します。")
        for p in candidate_paths:
            print(f"  - {os.path.abspath(p)}")

    else:
        print(f"MAFFT確認OK: {mafft_path}")
        print("=== 全自動一括解析を開始します ===")

        for t in targets:
            try:
                structure, csv_name, pdb_path = analyze_protein_interface(
                    t['id'], t['c1'], t['c2'], t['dna_list']
                )

                if structure is not None and csv_name is not None and pdb_path is not None:
                    fasta_name = f"seq_{t['id']}_{t['c1']}_{t['c2']}.fasta"
                    save_fasta_from_structure(t['id'], structure, [t['c1'], t['c2']], fasta_name)

                    blast_xml_name = f"blast_{t['id']}_{t['c1']}.xml"
                    run_blast_and_save_xml(fasta_name, blast_xml_name)

                    msa_input_fasta = f"msa_{t['id']}_{t['c1']}_{t['c2']}.fasta"
                    extract_hits_to_fasta(blast_xml_name, msa_input_fasta, fasta_name)

                    msa_output_fasta = f"aligned_{t['id']}_{t['c1']}_{t['c2']}.fasta"
                    input_path = os.path.abspath(msa_input_fasta)
                    print(f"--- {t['id']} の整列(MSA)を実行中... ---")
                    cmd = [mafft_path, "--auto", input_path]

                    with open(msa_output_fasta, "w") as out_f:
                        result = subprocess.run(
                            cmd, stdout=out_f, stderr=subprocess.PIPE, text=True, check=True
                        )
                    print(f"DONE: {t['id']} の整列が完了しました！ -> {msa_output_fasta}")

                    final_output_csv = f"final_interface_{t['id']}_{t['c1']}_{t['c2']}.csv"
                    query_id_in_fasta = f"{t['id']}_{t['c1']}"
                    integrate_conservation_to_csv(
                        msa_output_fasta=msa_output_fasta,
                        csv_name=csv_name,
                        output_csv=final_output_csv,
                        query_id=query_id_in_fasta
                    )

                    viz_pdb_name = f"viz_{t['id']}_{t['c1']}_{t['c2']}_cons.pdb"
                    save_conservation_to_pdb(
                        input_pdb=pdb_path,
                        integrated_csv=final_output_csv,
                        output_pdb=viz_pdb_name,
                        target_chain=t['c1']
                    )

                    viz_pml_name = f"viz_{t['id']}_{t['c1']}_{t['c2']}_cons.pml"
                    generate_pymol_script_protein(
                        pdb_filename=viz_pdb_name,
                        csv_file=final_output_csv, 
                        output_pml=viz_pml_name,
                        chain1=t['c1'],
                        chain2=t['c2']
                    )
                    
                    print(f"DONE: {t['id']} の全工程が正常に完了しました！")

            except subprocess.CalledProcessError as e:
                print(f"MAFFT実行エラー ({t['id']}): {e.stderr}")
            except Exception as e:
                print(f"ERROR: {t['id']} の処理中にエラーが発生しました: {e}")

        print("=== すべての工程が完了しました ===")

targetsが空です。タンパク質-タンパク質/DNA解析を実行する際は、上のtargetsに追加してください。
